# Multimodal messages

## Review

We used `create_agent` and shaped its answers with prompting.

* System prompts and few-shot examples
* Structured prompts and structured output with a Pydantic `response_format`

## Goals

So far, every message we sent was plain text.

[Multimodality](https://docs.langchain.com/oss/python/langchain/messages#multimodal) "refers to the ability to work with data that comes in different forms, such as text, audio, images, and video."

We'll send the agent text, an image, and a voice recording.

Not every model can read every kind of input.

The docs warn that "Not all models support all file types."

On Groq, `qwen/qwen3.8-27b` accepts images, so we use it for the text and image sections.

See the Groq [models](https://console.groq.com/docs/models) and [vision](https://console.groq.com/docs/vision) pages for what each model supports.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

We'll use [LangSmith](https://docs.langchain.com/langsmith/home) for [tracing](https://docs.langchain.com/langsmith/observability-concepts).

We'll log to the project set by `LANGSMITH_PROJECT` in the repo-root `.env`, which is `ai-engineering`.

## Text input

We start with an agent that has a system prompt, as in the prompting notebook.

In [2]:
from langchain.agents import create_agent

# qwen3.8 is the Groq chat model that accepts image input
agent = create_agent(
    model='groq:qwen/qwen3.8-27b',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

A message's content doesn't have to be a string.

It can be a list of [content blocks](https://docs.langchain.com/oss/python/langchain/messages#standard-content-blocks), each with a `type`.

A text block is the simplest one: `{"type": "text", "text": ...}`.

In [3]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

There is no capital of "The Moon" because the Moon is a natural satellite, not a sovereign nation or independent political entity. It does not have a government, population, or legal status that would designate a capital city.

However, since you are a science fiction writer, I can create a fictional capital for a future lunar colony. Here is a concept:

**Capital: Selene Prime**

*   **Location:** The Mare Imbrium (Sea of Rains) region, chosen for its flat terrain, relatively low seismic activity, and proximity to the Shackleton Crater (for water ice access).
*   **Description:** Selene Prime is a subterranean and surface hybrid city. The upper tier features geodesic domes with transparent aluminum plating, allowing natural starlight to filter in while regulating temperature and radiation. The lower tiers descend into the regolith, housing residential quarters, hydroponic farms, and manufacturing hubs.
*   **Governance:** It serves as the administrative center of the Lunar Autonomous 

## Image input

Now, let's send the agent an image.

Run the next cell and upload a PNG with the widget.

`resources/moon.png` works well.

In [4]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

Once a file is uploaded, `uploader.value` holds its name, type and raw bytes.

In [10]:
print(uploader.value)

({'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x700768163f40>, 'last_modified': datetime.datetime(2026, 9, 10, 2, 26, 29, 450000, tzinfo=datetime.timezone.utc)},)


Content blocks carry binary data as a base64 string, so we encode the uploaded bytes.

In [11]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

An image block holds the base64 data and its `mime_type`.

We send it in the same message as a text block, so the model sees both together.

In [12]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

Based on the visual elements in the image, here is a description of this capital city:

**Name:** Aethelgard Prime (or simply "The Spire City")

**Overview:**
Aethelgard Prime is not a sprawling metropolis that blankets the land, but rather a singular, monumental cluster of structures built into the jagged peaks of a desolate, rocky highland. It stands as a stark contrast to the barren, rust-colored terrain surrounding it—a beacon of advanced civilization on an otherwise inhospitable world.

**Architecture & Design:**
The city is defined by its towering, needle-like spires. These structures are sleek, metallic, and rise vertically with almost religious devotion to the sky. They appear to be constructed from dark, reflective alloys, punctuated by glowing cyan or teal light bands that suggest energy conduits, lighting, or perhaps defensive systems. The design is minimalist yet imposing—no ornamentation, only function and form. The tallest spire, likely the central administrative or comma

## Audio input

The standard content blocks cover audio too: `{"type": "audio", "base64": ..., "mime_type": "audio/wav"}`.

However, Groq has no chat model that accepts audio directly.

Instead, we'll transcribe the recording with Groq's [speech-to-text](https://console.groq.com/docs/speech-to-text) model, `whisper-large-v3-turbo`, and send the transcript as text.

The next cell records 5 seconds from your microphone and keeps the recording as WAV bytes.

In [13]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:05<00:00,  9.95it/s]

Done.


We transcribe the recording, then send the transcript to a new agent.

The message only contains text now, so we can go back to `openai/gpt-oss-20b`.

In [14]:
from groq import Groq

# Groq has no chat model that accepts audio, so transcribe it with Whisper first
transcription = Groq().audio.transcriptions.create(
    file=("audio.wav", wav_bytes),
    model="whisper-large-v3-turbo",
)

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": f"Tell me about this audio file. Transcript: {transcription.text}"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

I don’t have access to the actual audio file itself—just the transcript you supplied.  Based on that, here’s a quick rundown:

| Item | Details |
|------|---------|
| **Text** | “Hello, how are you doing?” |
| **Speaker** | One speaker (the transcript doesn’t indicate multiple speakers). |
| **Tone** | Friendly, conversational greeting. |
| **Content** | A simple, common opening line used in casual conversation or interviews. |
| **Length** | Roughly 4–5 seconds at normal speaking pace (about 6–8 words). |

Because I can’t analyze the waveform, I can’t provide:

* Exact duration or sampling rate  
* Audio quality (e.g., background noise, compression artifacts)  
* Speaker identity or acoustic characteristics  
* Prosodic features (pitch, intensity, timing)

If you need any of those details, you’ll have to share the audio file itself or a link to it.  Otherwise, let me know if you’d like a deeper linguistic or conversational analysis of the transcript!
